<a href="https://colab.research.google.com/github/Ravi-verma1498/Machine-Learning/blob/main/GNN_smiles_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install rdkit


import pandas as pd

import torch
from torch.utils.data import Dataset
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from rdkit import Chem


In [36]:
df = pd.read_csv("/content/BradleyDoublePlus-GoodMeltingPointDataset.csv")
df = df.drop(columns = ["Unnamed: 1","Unnamed: 3","Unnamed: 4","Unnamed: 5","Unnamed: 7","Unnamed: 6"])
print(f"Cleaned dataset size: {len(df)}")

Cleaned dataset size: 3041


In [35]:
from rdkit.Chem import SanitizeFlags
def is_valid_smiles(smiles):
    """Check if SMILES is valid with aromatic handling"""
    mol = Chem.MolFromSmiles(smiles, sanitize=False)
    if mol is None:
        return False
    try:
        # Skip kekulization for aromatic systems


        Chem.SanitizeMol(mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_KEKULIZE)
        return True
    except:
        return False

df_clean = df[df['smiles'].apply(is_valid_smiles)].reset_index(drop=True)
print(f"Cleaned dataset size: {len(df_clean)}")

Cleaned dataset size: 3041


In [37]:
def atomf(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetHybridization().real,
        atom.GetDegree(),
        int(atom.GetIsAromatic()),
        atom.GetTotalValence()
    ]


def smiles_to_graph(smile, target):
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        print(f"Warning: Invalid SMILES string '{smile}'")
        return None

    node_features = [atomf(atom) for atom in mol.GetAtoms()]
    edge_attr = []
    edge_index = []

    for bond in mol.GetBonds():
        start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.append([start, end])
        edge_index.append([end, start])  # Bidirectional edges
        edge_attr.append([bond.GetBondTypeAsDouble()])
        edge_attr.append([float(bond.GetIsConjugated())])  # Ensure float type

    return Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).T,
        edge_attr=torch.tensor(edge_attr, dtype=torch.float),
        y=torch.tensor([target], dtype=torch.float),  # Make sure target is a tensor of shape [1]
    )


Cleaned dataset size: 0


In [38]:
from torch_geometric.loader import DataLoader  # PyG's DataLoader


class BoilingPointDataset(Dataset):
    def __init__(self, df):
        super().__init__()
        self.df = df


    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        graph_data = smiles_to_graph(row["smiles"], row["mpC"])
        if graph_data is None:
            return None
        return graph_data



dataset = BoilingPointDataset(df_clean)
dataset = [data for data in dataset if data is not None]

train_size = int(0.8 * len(dataset))
train_data, test_data = random_split(dataset, [train_size, len(dataset) - train_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

[09:35:48] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[09:35:48] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[09:35:48] Can't kekulize mol.  Unkekulized atoms: 24 25 26 27 28 31 32 33 34


[09:35:48] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[09:35:48] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[09:35:48] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 8 9


[09:35:48] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 20 21 22 23 24


[09:35:49] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 8 9 10 11
[09:35:49] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 8


[09:35:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8


[09:35:50] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 8 9


[09:35:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[09:35:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 12 13 14 15 16


[09:35:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[09:35:51] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7
[09:35:51] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 8 9


In [30]:
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_dim, out_dim):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.fc = torch.nn.Linear(hidden_dim, out_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))


        x = global_mean_pool(x, data.batch)
        x = self.fc(x)

        return x



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model, optimizer and loss function
model = GNN(in_channels=5, hidden_dim=64, out_dim=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.MSELoss()

In [40]:

from torch_geometric.loader import DataLoader  # PyG's DataLoader

def train(model, train_loader):
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = loss_fn(out.squeeze(), data.y)  # Squeeze to match shapes
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

# Training loop
for epoch in range(200):
    loss = train(model, train_loader)
    print(f"Epoch {epoch + 1}: Loss={loss:.4f}")




Epoch 1: Loss=4545.1337
Epoch 2: Loss=4701.5063
Epoch 3: Loss=4589.1738
Epoch 4: Loss=4355.8824
Epoch 5: Loss=4617.9778
Epoch 6: Loss=4341.6839
Epoch 7: Loss=4433.8813
Epoch 8: Loss=4494.4165
Epoch 9: Loss=4565.6627
Epoch 10: Loss=4327.1845
Epoch 11: Loss=4323.9078
Epoch 12: Loss=4281.9018
Epoch 13: Loss=4307.1024
Epoch 14: Loss=4046.9330
Epoch 15: Loss=4183.3581
Epoch 16: Loss=4148.5230
Epoch 17: Loss=3987.1056
Epoch 18: Loss=4256.3211
Epoch 19: Loss=4196.8295
Epoch 20: Loss=4172.5850
Epoch 21: Loss=4043.7587
Epoch 22: Loss=3966.6335
Epoch 23: Loss=3942.5604
Epoch 24: Loss=4270.3863
Epoch 25: Loss=3888.5185
Epoch 26: Loss=4117.2508
Epoch 27: Loss=3992.3040
Epoch 28: Loss=3918.6297
Epoch 29: Loss=3880.0149
Epoch 30: Loss=3812.9979
Epoch 31: Loss=4201.8381
Epoch 32: Loss=3894.7628
Epoch 33: Loss=3819.7752
Epoch 34: Loss=3683.9138
Epoch 35: Loss=3801.1230
Epoch 36: Loss=3733.3991
Epoch 37: Loss=4330.4281
Epoch 38: Loss=3705.0498
Epoch 39: Loss=3801.3102
Epoch 40: Loss=3765.7307
Epoch 41:

In [42]:
def test(test_loader, model):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            out = model(data).squeeze()
            loss = loss_fn(out, data.y)
            total_loss += loss.item()
    return total_loss / len(test_loader)

# Evaluate the model on the test set
test_loss = test(test_loader, model)
print(f"Test Loss: {test_loss:.4f}")

Test Loss: 2580.0681
